<a href="https://colab.research.google.com/github/Fespinoza1477/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f"Count of Pop-Tart rows: {len(pop_tarts)}")
print(pop_tarts)

Count of Pop-Tart rows: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
strawberry_pop_tarts = pop_tarts[pop_tarts['product_name'].str.lower().str.contains('straw')]
print("Strawberry Pop-Tarts:")
print(strawberry_pop_tarts[['sku', 'product_name']])

other_pop_tarts = pop_tarts[~pop_tarts['product_name'].str.lower().str.contains('straw')]
print("\nNon-Strawberry Pop-Tarts:")
print(other_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tarts:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Non-Strawberry Pop-Tarts:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
sku_mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])
print(cat[['sku', 'canonical_sku', 'product_name']])

             sku canonical_sku          product_name
0          PT-12         PT-12  pop-tarts strawberry
1     SKU#459812         PT-12   Strawberry Pop-Tart
2  POPTART-STRAW         PT-12  Pop-Tarts Strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print("---Distinct Categories Before ---")
print(cat['category'].value_counts())
cat['category_norm'] = cat['category'].str.strip().str.title()

print("\n--- Distinct Categories After ---")
print(cat['category_norm'].value_counts())

---Distinct Categories Before ---
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

--- Distinct Categories After ---
category_norm
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
import string
def normalize_string(text):
  clean = text.lower().translate(str.maketrans('', '', string.punctuation))
  return ' '.join(sorted(clean.split()))

cat['sort_key'] = cat['product_name'].apply(normalize_string)
fragmented_groups = cat.groupby('sort_key').filter(lambda x: len(x) >1)

print(fragmented_groups[['sku', 'product_name', 'sort_key']].sort_values('sort_key'))

             sku          product_name             sort_key
0          PT-12  pop-tarts strawberry  poptarts strawberry
2  POPTART-STRAW  Pop-Tarts Strawberry  poptarts strawberry


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert cat['canonical_sku'].isnull().sum() == 0, "Error: Nulls were introduced during SKU mapping."

assert len(cat) == 3, "Error: Row count changed during consolidation."

print("Assertions passed: Consolidation renamed values without adding or removing rows.")

Assertions passed: Consolidation renamed values without adding or removing rows.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

1. Coca-Cola Fragmented as "Coke 12-pack", "Coca Cola 12pk", and "Coke 12 cans". I would group these by brand ("Coca-cola"), container type ("Can") and pack size ("12-pack").
2. Bananas Fragmented as "Organic Banana", "Bananas Org", and "Banana". I would group these by separating standard from organic, creating one canonical SKU for conventional bananas and one for organic bananas.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Consolidation stops being a data cleaning task and becomes a business decision when two items share the same physical product but differ in metadata that affects pricing, promotions, or vendor tracking. Ex. Are "Holiday Edition Strawberry Pop-Tarts" the exact same product as "Regular Strawberry Pop-Tarts"? They taste identical, but the holiday ones might need to be aggressively clearenced after December 25th. I would ask the Merchandising or Inventory Planning team before combining them.

_your answer here_